In [1]:
import os
os.environ['TORCHDYNAMO_VERBOSE'] = '1'

from torch.optim import Adam
from tqdm import trange, tqdm
import torch

device = torch.device('cuda:0')

In [37]:
import torch
from torch import nn
from torch.nn import functional as F
from torch.nn.attention.flex_attention import create_block_mask, flex_attention

flex_attention_ = torch.compile(flex_attention, dynamic=True)

class Flextension(nn.Module):
    def __init__(self, in_proj_weight, in_proj_bias, out_proj_weight, out_proj_bias, block_mask, attn_heads):
        super().__init__()
        # weight and bias for linear layers
        self.in_proj_weight = in_proj_weight
        self.in_proj_bias = in_proj_bias
        self.out_proj_weight = out_proj_weight
        self.out_proj_bias = out_proj_bias
        # attributes
        self.attn_heads = attn_heads
        self.batch_first = True
        self._qkv_same_embed_dim = True

        # attributes for flex_attention
        self.kernel_options = {"BLOCK_M": 32, "BLOCK_N": 32, "BLOCK_M1": 16, "BLOCK_N1": 32, "BLOCK_M2": 32,
                               "BLOCK_N2": 16, }
        self.block_mask = block_mask

    def forward(self, x, x1, x2, attn_mask=None, key_padding_mask=None, need_weights=False, is_causal=False):
        q, k, v = F.linear(x, self.in_proj_weight, bias=self.in_proj_bias).chunk(3, -1)
        q_ = q.unflatten(-1, (self.attn_heads, -1)).transpose(2, 1)

        k_ = k.unflatten(-1, (self.attn_heads, -1)).transpose(2, 1)
        v_ = v.unflatten(-1, (self.attn_heads, -1)).transpose(2, 1)
        y_ = flex_attention_(q_, k_, v_, kernel_options=self.kernel_options, block_mask=self.block_mask)
        y = y_.transpose(2, 1).flatten(2, -1)
        y = F.linear(y, self.out_proj_weight, bias=self.out_proj_bias)
        return y


class FlexFormer(nn.TransformerEncoderLayer):
    def __init__(self, in_channel: int, nhead: int, dim_ff_scale: int = 4, dropout: float = 0.1, activation=F.leaky_relu, block_mask=None):
        super().__init__(d_model=in_channel, nhead=nhead, dim_feedforward=in_channel * dim_ff_scale, dropout=dropout,
                         activation=activation, layer_norm_eps=1e-5, batch_first=True, norm_first=True)
        self.self_attn = Flextension(self.self_attn.in_proj_weight, self.self_attn.in_proj_bias,
                                     self.self_attn.out_proj.weight, self.self_attn.out_proj.bias, block_mask, attn_heads=nhead)


class FlexBlock(nn.Module):
    def __init__(self, patch_size: torch.Tensor, kernel: int, in_channel: int, nhead: int, device: str, n_layers:int):
        super().__init__()
        print(n_layers, 'x', in_channel, patch_size.tolist())
        assert (patch_size % 2 == 0).all(), "Patch size must be even"
        self.kernel = torch.tensor([kernel, kernel], device=device)
        self.patch_size = patch_size.to(device)
        S = patch_size.prod().item()
        block_mask = create_block_mask(self.compute_mask, None, nhead, S, S, device, _compile=True)

        self.layers = nn.ModuleList([FlexFormer(in_channel, nhead, block_mask=block_mask) for _ in range(n_layers)])

    def forward(self, x):
        identity = x
        for layer in self.layers:
            x = layer(x)
        x += identity
        return x

    def compute_mask(self, b, h, q_idx, kv_idx):
        # unravel index
        q_x = q_idx % self.patch_size[1]
        q_y = q_idx // self.patch_size[1]
        kv_x = kv_idx % self.patch_size[1]
        kv_y = kv_idx // self.patch_size[1]

        # compute mask
        is_valid_x = (q_x - kv_x).abs() <= self.kernel[0] // 2
        is_valid_y = (q_y - kv_y).abs() <= self.kernel[1] // 2
        is_valid = is_valid_x & is_valid_y
        return is_valid

In [20]:
patch_size = torch.tensor([16]*2)
input_channels = 128

model_block = FlexBlock(patch_size, 3, input_channels, 4, 'cuda', n_layers=4)

4 x 128 [16, 16]


In [21]:
from copy import deepcopy

x = torch.randn(8, patch_size.prod().item(), input_channels).to(device)
y = torch.randn_like(x)

model = deepcopy(model_block).to(device)
optimizer = Adam(model.parameters(), lr=1e-3)
for i in trange(5000, disable=True):
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        y_hat = model(x)
        loss = F.mse_loss(y_hat, y)
    loss.backward()
    optimizer.step()
    if i % 500 == 0:
        tqdm.write(f"iter: {i}, loss: {loss.item():.4f}")

iter: 0, loss: 5.4144
iter: 500, loss: 0.8228
iter: 1000, loss: 0.7178


SystemError: <method 'dim' of 'torch._C.TensorBase' objects> returned a result with an exception set

In [22]:
y = torch.randint(0, 4, (8,)).to(device)

model = deepcopy(model_block).to(device)
classifier = nn.Linear(input_channels, 4, bias=False).to(device)
optimizer = Adam(list(classifier.parameters()) + list(model.parameters()), lr=1e-3)
for i in trange(5000, disable=True):
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        y_hat = model(x)
        y_hat = classifier(y_hat.mean(1))
        loss = F.cross_entropy(y_hat, y)
    loss.backward()
    optimizer.step()
    if i % 500 == 0:
        tqdm.write(f"iter: {i}, loss: {loss.item():.4f}, acc: {(y_hat.argmax(-1) == y).float().mean().item():.4f}")

iter: 0, loss: 1.4756, acc: 0.2500
iter: 500, loss: 0.0000, acc: 1.0000


Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x70d069104c20>>
Traceback (most recent call last):
  File "/home/keuth/miniforge3/envs/FlexConvNightly/lib/python3.13/site-packages/ipykernel/ipkernel.py", line 775, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(
KeyboardInterrupt: 


KeyboardInterrupt: 

In [38]:
from typing import List


class PoolWithChannelExpansion(nn.Module):
    def __init__(self, patch_size, in_channel, out_channel, kernel_size, stride, padding, pool_op='avg'):
        super().__init__()
        self.patch_size = patch_size
        self.unflatten = nn.Unflatten(2, patch_size.tolist())
        if pool_op == 'avg':
            self.pool = nn.Sequential(
                nn.AvgPool2d(kernel_size, stride, padding),
                nn.Conv2d(in_channel, out_channel, 1, 1, 0)
            )
        elif pool_op == 'max':
            self.pool = nn.Sequential(
                nn.MaxPool2d(kernel_size, stride, padding),
                nn.Conv2d(in_channel, out_channel, 1, 1, 0)
            )
        elif pool_op == 'conv_block':
            self.pool = nn.Sequential(
                nn.Conv2d(in_channel, out_channel, kernel_size, stride, padding, bias=False, groups=in_channel),
                nn.InstanceNorm2d(out_channel, affine=True),
                nn.LeakyReLU()
            )
        elif pool_op == 'conv':
            self.pool = nn.Conv2d(in_channel, out_channel, kernel_size, stride, padding, bias=False, groups=in_channel)
        else:
            raise ValueError(f"Unknown pooling operation: {pool_op}")

    def forward(self, x):
        # x (B, H*W, C)
        x = x.permute(0, 2, 1)  # (B, C, H*W)
        x = self.unflatten(x)  # (B, C, H, W)
        y = self.pool(x)
        y_ = y.flatten(2, 3)  # (B, C', H*W)
        y_ = y_.permute(0, 2, 1)  # (B, H*W, C')
        return y_


class FlexNetAvgPoolModel(nn.Module):
    def __init__(self, n_channel: int, n_classes: int, n_heads: int = 4,
                 patch_size: List[int] = [224, 224], pool_op: str = 'conv_block',
                 device: str = 'cuda'):
        super().__init__()
        patch_size = torch.tensor(patch_size)
        flexblock_kwargs = dict(nhead=n_heads, device=device)

        n_out_channels = 64
        self.first_layer = nn.Sequential(
            PoolWithChannelExpansion(patch_size.clone(), n_channel, n_out_channels, 7, 2, 3),
            FlexBlock(patch_size.clone() // 2, 3, n_out_channels, n_layers=1, **flexblock_kwargs),
        )
        patch_size //= 2

        self.layers = nn.ModuleList()
        for i, repeats in enumerate([2, 2, 2, 2]): # [2, 2, 2, 2]
            if i == 0:  # max pool of second group
                self.layers.append(
                    PoolWithChannelExpansion(patch_size.clone(), n_out_channels, n_out_channels, 3, 2, 1, 'max'))
            else:
                self.layers.append(
                    PoolWithChannelExpansion(patch_size.clone(), n_out_channels, n_out_channels * 2, 3, 2, 1, pool_op))
                n_out_channels *= 2
            patch_size //= 2
            print(f'Group {i + 1}: {patch_size.tolist()} with {n_out_channels} channels')
            self.layers.append(
                FlexBlock(patch_size.clone(), 3, n_out_channels, n_layers=2 * repeats, **flexblock_kwargs),
            )
            self.classifier = nn.Linear(n_out_channels, n_classes)

    def forward(self, x):
        # view for transformer
        x_ = x.flatten(2).permute(0, 2, 1).contiguous()  # (B, C, H*W)
        x_ = self.first_layer(x_)
        for i, layer in enumerate(self.layers):
            x_ = layer(x_)
        # avg pooling
        x_ = x_.mean(1)
        y_hat = self.classifier(x_)
        return y_hat

In [39]:
y = torch.randint(0, 4, (8,)).to(device)

x = torch.randn(8, 1, 128, 128).to(device)

model = FlexNetAvgPoolModel(1, 4, patch_size=[128]*2).to(device)
#model = torch.compile(model, dynamic=False)

1 x 64 [64, 64]
Group 1: [32, 32] with 64 channels
4 x 64 [32, 32]
Group 2: [16, 16] with 128 channels
4 x 128 [16, 16]
Group 3: [8, 8] with 256 channels
4 x 256 [8, 8]
Group 4: [4, 4] with 512 channels
4 x 512 [4, 4]


In [41]:
optimizer = Adam(model.parameters(), lr=1e-4)
for i in trange(10000, disable=True):
    optimizer.zero_grad()
    with torch.autocast(device_type='cuda', dtype=torch.bfloat16):
        y_hat = model(x)
        loss = F.cross_entropy(y_hat, y)
    loss.backward()
    optimizer.step()
    if i % 50 == 0:
        tqdm.write(f"iter: {i}, loss: {loss.item():.4f}, acc: {(y_hat.argmax(-1) == y).float().mean().item():.4f}")

# todo add positional embedding

iter: 0, loss: 0.0002, acc: 1.0000
iter: 50, loss: 0.0002, acc: 1.0000
iter: 100, loss: 0.0001, acc: 1.0000


KeyboardInterrupt: 

In [44]:
print(model(torch.randn(8, 1, 128, 128).to(device)).shape)
print(model(torch.randn(2, 1, 128, 128).to(device)).shape)
print(model(torch.randn(6, 1, 128, 128).to(device)).shape)

torch.Size([8, 4])
torch.Size([2, 4])
torch.Size([6, 4])
